# 1D optimal scheduling reference

This notebook is a clean analytical check of the optimal scheduling hypothesis from Tsimpos, Ren, Zech, and Marzouk, [*Optimal Scheduling of Dynamic Transport*](https://proceedings.mlr.press/v291/tsimpos25a.html) (COLT 2025, arXiv:2504.14425). The hypothesis tested here is not that the schedule changes the exact transport endpoint. It cannot: every admissible schedule with $\tau(0)=0$ and $\tau(1)=1$ reaches the same monotone rearrangement $T$ at the final time.

Instead, the question is numerical and dynamical: does the closed-form optimal schedule $\tau_\infty$ flatten the spatial Lipschitz constant of the ODE velocity field, and does that flatter profile translate into smaller finite-step ODE integration error than the identity schedule $\tau(t)=t$ at the same computational budget?

Everything here is one-dimensional and reference-only. The transport map is the monotone rearrangement $T = F_\nu^{-1} \circ F_\mu$. There are no neural networks, no training loops, no learned vector fields, and no external datasets.


## Setup

The paper assumes compact domains. For the unbounded examples below, the numerical experiment uses high-probability compact grids and explicitly records the grid resolution. The Lipschitz calculation uses the one-dimensional form of Eq. (9):

$$
L_\tau(t) = \sup_x \left|\dot\tau(t)\,\frac{T'(x)-1}{1 + \tau(t)(T'(x)-1)}\right|.
$$

This avoids a separate numerical inversion of $X_\tau(\cdot,t)$ for the Lipschitz diagnostic, while the velocity itself is still $v_\tau(y,t)=\dot\tau(t)(T(x)-x)$ with $y=X_\tau(x,t)$.

## Schedule-invariant endpoint

For a source point $x$, the scheduled interpolation is

$$
X_\tau(x,t)=(1-\tau(t))x+\tau(t)T(x).
$$

Because both schedules used below satisfy $\tau(1)=1$, the exact final map is

$$
X_\tau(x,1)=T(x).
$$

This identity is the main guardrail for the numerical experiment. The exact endpoint distribution is schedule-invariant, so an exact final Wasserstein distance cannot be used as evidence that one schedule is better. The meaningful comparison is whether a finite-step ODE solver follows the scheduled dynamics more accurately when the spatial Lipschitz profile is flatter.


In [ ]:
from dataclasses import dataclass
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import optimize, special, stats

# Keep the visual style compact and consistent across all diagnostic plots.
plt.rcParams.update(
    {
        "figure.figsize": (6, 4),
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.frameon": False,
    }
)

# Shared colors: identity schedule in blue, optimal schedule in red.
BLUE = "steelblue"
RED = "crimson"
GRAY = "0.35"
GOLD = "darkorange"

# The probability clipping avoids infinite quantiles at distribution endpoints.
PROB_EPS = 1e-14
T_GRID = np.linspace(0.0, 1.0, 401)
SUMMARY_ROWS = []
CASE_RESULTS = []


def clip_prob(u: np.ndarray, eps: float = PROB_EPS) -> np.ndarray:
    """Clip probabilities away from 0 and 1 before calling inverse CDFs.

    Many of the examples use Gaussian or Laplace quantiles, which diverge at
    exact probabilities 0 and 1. A tiny clip keeps the reference computations
    finite without changing the interior behavior being studied.
    """
    return np.clip(np.asarray(u, dtype=float), eps, 1.0 - eps)


In [ ]:
@dataclass
class Distribution1D:
    """Container for the scalar distribution operations used in the notebook.

    The transport map only needs density, log-density, CDF, and quantile
    functions. Keeping this as a small protocol-like dataclass makes the later
    code independent of whether the distribution comes from SciPy or a custom
    mixture implementation.
    """

    name: str
    pdf: Callable[[np.ndarray], np.ndarray]
    cdf: Callable[[np.ndarray], np.ndarray]
    ppf: Callable[[np.ndarray], np.ndarray]
    logpdf: Callable[[np.ndarray], np.ndarray]


def scipy_distribution(name: str, frozen_dist) -> Distribution1D:
    """Wrap a SciPy frozen distribution in the local Distribution1D interface."""
    return Distribution1D(
        name=name,
        pdf=frozen_dist.pdf,
        cdf=frozen_dist.cdf,
        ppf=frozen_dist.ppf,
        logpdf=frozen_dist.logpdf,
    )


class GaussianMixture1D:
    """Small analytic one-dimensional Gaussian mixture helper.

    SciPy does not expose a frozen distribution object for arbitrary Gaussian
    mixtures with a quantile function, so this class implements the four methods
    needed by Distribution1D. The quantile is computed by scalar root finding.
    """

    def __init__(self, weights, means, stds, name="Gaussian mixture"):
        """Store normalized mixture parameters and a safe quantile bracket.

        The bracket spans many component standard deviations plus a margin so
        Brent's method can solve CDF(z) = u for all clipped probabilities used
        by the notebook.
        """
        self.weights = np.asarray(weights, dtype=float)
        self.weights = self.weights / self.weights.sum()
        self.means = np.asarray(means, dtype=float)
        self.stds = np.asarray(stds, dtype=float)
        self.name = name

        # Wide but finite bracket for scalar inverse-CDF root solves.
        self._lo = float(np.min(self.means - 14.0 * self.stds) - 1.0)
        self._hi = float(np.max(self.means + 14.0 * self.stds) + 1.0)

    def pdf(self, x):
        """Evaluate the mixture density at scalar or array inputs."""
        x = np.asarray(x, dtype=float)
        out = np.zeros_like(x, dtype=float)
        for weight, mean, std in zip(self.weights, self.means, self.stds):
            out += weight * stats.norm.pdf(x, loc=mean, scale=std)
        return out

    def logpdf(self, x):
        """Evaluate the mixture log-density using log-sum-exp for stability."""
        x = np.asarray(x, dtype=float)
        terms = [
            np.log(weight) + stats.norm.logpdf(x, loc=mean, scale=std)
            for weight, mean, std in zip(self.weights, self.means, self.stds)
        ]
        return special.logsumexp(np.stack(terms, axis=0), axis=0)

    def cdf(self, x):
        """Evaluate the mixture CDF as the weighted sum of component CDFs."""
        x = np.asarray(x, dtype=float)
        out = np.zeros_like(x, dtype=float)
        for weight, mean, std in zip(self.weights, self.means, self.stds):
            out += weight * stats.norm.cdf(x, loc=mean, scale=std)
        return out

    def _ppf_scalar(self, u: float) -> float:
        """Invert the mixture CDF for one clipped probability value."""
        u = float(np.clip(u, PROB_EPS, 1.0 - PROB_EPS))
        return optimize.brentq(
            lambda z: float(self.cdf(z) - u),
            self._lo,
            self._hi,
            xtol=1e-12,
            rtol=1e-12,
            maxiter=200,
        )

    def ppf(self, u):
        """Vectorized inverse CDF for scalar or array probabilities."""
        u = clip_prob(u)
        flat = np.ravel(u)

        # Brent's method is scalar, so flatten and reshape to preserve input shape.
        out = np.array([self._ppf_scalar(float(ui)) for ui in flat])
        return out.reshape(np.shape(u))

    @property
    def as_distribution(self) -> Distribution1D:
        """Expose the mixture through the same interface as SciPy distributions."""
        return Distribution1D(
            name=self.name,
            pdf=self.pdf,
            cdf=self.cdf,
            ppf=self.ppf,
            logpdf=self.logpdf,
        )


In [ ]:
@dataclass
class CaseSpec:
    """All distribution and grid choices for one 1D transport experiment.

    The main grid is used for plotting, while mode_y and extra_u inject points
    that are important for estimating extrema of T'(x) on difficult examples.
    """

    name: str
    source: Distribution1D
    target: Distribution1D
    x_grid: np.ndarray
    x0_grid: np.ndarray
    mode_y: tuple[float, ...] = ()
    extra_u: tuple[float, ...] = ()
    log_clip: float = 50.0
    gaussian_ratio: float | None = None


def quantile_grid(dist: Distribution1D, q_low: float, q_high: float, n: int) -> np.ndarray:
    """Build a deterministic grid by evaluating equally spaced quantiles."""
    q = np.linspace(q_low, q_high, n)
    return np.asarray(dist.ppf(clip_prob(q)), dtype=float)


def sorted_unique(values: np.ndarray) -> np.ndarray:
    """Return sorted finite grid values with near-duplicates removed."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    # Rounding removes tiny duplicate differences introduced by inverse CDF calls.
    return np.unique(np.round(np.sort(values), decimals=14))


def transport_values(case: CaseSpec, x: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Evaluate the monotone transport map and its derivative on source points.

    In one dimension, T(x) = F_target^{-1}(F_source(x)). Its derivative follows
    from differentiating the CDF identity: T'(x) = p_source(x) / p_target(T(x)).
    """
    u = clip_prob(case.source.cdf(x))
    tx = case.target.ppf(u)
    log_tprime = case.source.logpdf(x) - case.target.logpdf(tx)

    # Clip only the exponentiation step; the unclipped log derivative is kept too.
    tprime = np.exp(np.clip(log_tprime, -case.log_clip, case.log_clip))
    return tx, tprime, log_tprime


def make_evaluation_grid(case: CaseSpec) -> np.ndarray:
    """Combine plotting, modal, and special-probability points for diagnostics."""
    pieces = [case.x_grid]
    if case.mode_y:
        # Pull target modes back to source coordinates so extrema near modes are sampled.
        mode_y = np.asarray(case.mode_y, dtype=float)
        mode_u = clip_prob(case.target.cdf(mode_y))
        pieces.append(case.source.ppf(mode_u))
    if case.extra_u:
        # Add case-specific source quantiles near splits or tails.
        pieces.append(case.source.ppf(clip_prob(np.asarray(case.extra_u, dtype=float))))
    return sorted_unique(np.concatenate(pieces))


In [ ]:
def optimal_schedule_factory(sigma_min: float, sigma_max: float, atol: float = 1e-10):
    """Construct the closed-form tau_infty schedule and metadata.

    In 1D, sigma_min and sigma_max are numerical extrema of T'(x). The
    transition branch is written in the equivalent integral form: if
    A(tau) = max{f*/(1 + tau f*), -g*/(1 + tau g*)}, then the optimizer
    satisfies tau_dot(t) A(tau(t)) = int_0^1 A(s) ds.
    """
    sigma_min = float(sigma_min)
    sigma_max = float(sigma_max)
    if sigma_min <= 0 or sigma_max <= 0:
        raise ValueError("T' extrema must be positive.")

    # If T' is numerically constant, the simple one-sided formula is exact.
    if abs(sigma_max - sigma_min) <= atol:
        sigma = 0.5 * (sigma_min + sigma_max)
        return simple_schedule_factory(sigma), {"case": "constant", "t0": np.nan}

    f_star = sigma_max - 1.0
    g_star = sigma_min - 1.0

    # Pure contraction and pure expansion do not need the transition formula.
    if sigma_max <= 1.0 + atol:
        return simple_schedule_factory(sigma_min), {"case": "contraction", "t0": np.nan}
    if sigma_min >= 1.0 - atol:
        return simple_schedule_factory(sigma_max), {"case": "expansion", "t0": np.nan}

    # Mixed contraction/expansion: locate the tau value where the active bound switches.
    tau0 = -0.5 * (1.0 / f_star + 1.0 / g_star)
    if not (0.0 <= tau0 <= 1.0):
        if f_star >= -g_star:
            return simple_schedule_factory(sigma_max), {"case": "expansion", "t0": np.nan}
        return simple_schedule_factory(sigma_min), {"case": "contraction", "t0": np.nan}

    left_mass = np.log1p(f_star * tau0)
    right_mass = -np.log(sigma_min) + np.log1p(g_star * tau0)
    lambda_star = left_mass + right_mass
    t0 = left_mass / lambda_star
    log_right_prefactor = np.log1p(g_star * tau0) + left_mass

    def schedule(t):
        """Evaluate the mixed optimal schedule and its time derivative."""
        t = np.asarray(t, dtype=float)
        left = t <= t0
        tau = np.empty_like(t, dtype=float)
        tau_dot = np.empty_like(t, dtype=float)

        # Expansion-dominated branch before the switch time.
        exp_left = np.exp(lambda_star * t[left])
        tau[left] = (exp_left - 1.0) / f_star
        tau_dot[left] = lambda_star * exp_left / f_star

        # Contraction-dominated branch after the switch time.
        exp_right = np.exp(log_right_prefactor - lambda_star * t[~left])
        tau[~left] = (exp_right - 1.0) / g_star
        tau_dot[~left] = -lambda_star * exp_right / g_star

        return np.clip(tau, 0.0, 1.0), np.maximum(tau_dot, 0.0)

    return schedule, {"case": "transition", "t0": float(t0)}


def simple_schedule_factory(sigma: float):
    """Construct the one-sided optimal schedule for pure contraction/expansion.

    The returned callable maps t to (tau(t), tau_dot(t)). When sigma is one,
    the optimal schedule reduces to the identity schedule.
    """
    sigma = float(sigma)
    if abs(sigma - 1.0) < 1e-10:
        def identity(t):
            """Evaluate tau(t)=t and tau_dot(t)=1."""
            t = np.asarray(t, dtype=float)
            return t, np.ones_like(t)

        return identity

    log_sigma = np.log(sigma)

    def schedule(t):
        """Evaluate the exponential schedule that equalizes one-sided stiffness."""
        t = np.asarray(t, dtype=float)
        sigma_t = np.exp(log_sigma * t)
        tau = (sigma_t - 1.0) / (sigma - 1.0)
        tau_dot = log_sigma * sigma_t / (sigma - 1.0)
        return np.clip(tau, 0.0, 1.0), np.maximum(tau_dot, 0.0)

    return schedule


def identity_schedule(t):
    """Evaluate the baseline schedule tau(t)=t and tau_dot(t)=1."""
    t = np.asarray(t, dtype=float)
    return t, np.ones_like(t)


In [ ]:
def lipschitz_curve(t_grid: np.ndarray, tprime: np.ndarray, schedule) -> np.ndarray:
    """Compute the empirical spatial Lipschitz curve L_tau(t) on a grid.

    The formula uses only T'(x), tau(t), and tau_dot(t), avoiding numerical
    inversion for the Lipschitz diagnostic itself.
    """
    tau, tau_dot = schedule(t_grid)
    delta = tprime[None, :] - 1.0
    denom = 1.0 + tau[:, None] * delta

    # Broadcast over times and source-grid points, then maximize over space.
    values = np.abs(tau_dot[:, None] * delta / denom)
    return np.nanmax(values, axis=1)


def flow_positions(x0: np.ndarray, tx0: np.ndarray, schedule, t_grid: np.ndarray) -> np.ndarray:
    """Evaluate exact scheduled trajectories for selected source particles."""
    tau, _ = schedule(t_grid)
    return (1.0 - tau[:, None]) * x0[None, :] + tau[:, None] * tx0[None, :]


def velocity_on_reference_grid(x: np.ndarray, tx: np.ndarray, schedule, t: float):
    """Evaluate y=X_tau(x,t) and v_tau(y,t) on reference coordinates."""
    tau, tau_dot = schedule(np.asarray([t], dtype=float))
    y = (1.0 - tau[0]) * x + tau[0] * tx
    v = tau_dot[0] * (tx - x)
    return y, v


def run_case(case: CaseSpec):
    """Run all analytical schedule diagnostics for one case specification."""
    x_eval = make_evaluation_grid(case)
    tx_eval, tprime_eval, log_tprime_eval = transport_values(case, x_eval)

    # The optimal schedule depends only on the empirical extrema of T'(x).
    sigma_min = float(np.min(tprime_eval))
    sigma_max = float(np.max(tprime_eval))
    schedule_opt, schedule_info = optimal_schedule_factory(sigma_min, sigma_max)

    l_id = lipschitz_curve(T_GRID, tprime_eval, identity_schedule)
    l_opt = lipschitz_curve(T_GRID, tprime_eval, schedule_opt)

    # Separate plotting data keeps figures smooth while diagnostics use enriched grids.
    tx_plot, tprime_plot, log_tprime_plot = transport_values(case, case.x_grid)
    tx0, _, _ = transport_values(case, case.x0_grid)

    result = {
        "case": case,
        "x_eval": x_eval,
        "tx_eval": tx_eval,
        "tprime_eval": tprime_eval,
        "log_tprime_eval": log_tprime_eval,
        "tx_plot": tx_plot,
        "tprime_plot": tprime_plot,
        "log_tprime_plot": log_tprime_plot,
        "sigma_min": sigma_min,
        "sigma_max": sigma_max,
        "schedule_opt": schedule_opt,
        "schedule_info": schedule_info,
        "l_id": l_id,
        "l_opt": l_opt,
        "tx0": tx0,
    }

    # Store both full results and compact rows for later summary tables.
    CASE_RESULTS.append(result)
    SUMMARY_ROWS.append(
        {
            "case": case.name,
            "Lambda[id]": float(np.max(l_id)),
            "Lambda[tau_infty]": float(np.max(l_opt)),
            "ratio": float(np.max(l_id) / np.max(l_opt)),
            "sigma_min": sigma_min,
            "sigma_max": sigma_max,
            "schedule_case": schedule_info["case"],
            "t0": schedule_info["t0"],
        }
    )
    return result


def extrema_table(result):
    """Display the grid extrema that determine the optimal schedule."""
    row = {
        "grid points": len(result["x_eval"]),
        "sigma_min": result["sigma_min"],
        "sigma_max": result["sigma_max"],
        "schedule": result["schedule_info"]["case"],
        "t0": result["schedule_info"]["t0"],
        "max log T' on grid": float(np.max(result["log_tprime_eval"])),
    }
    display(pd.DataFrame([row]))


In [ ]:
def plot_tprime(result):
    """Plot the transport derivative T'(x) for one case."""
    case = result["case"]
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.semilogy(case.x_grid, result["tprime_plot"], color=GRAY, lw=1.8)
    ax.set_title(f"{case.name}: transport derivative")
    ax.set_xlabel("source coordinate x")
    ax.set_ylabel("T'(x)")
    plt.show()


def plot_trajectories(result):
    """Plot exact particle trajectories under identity and optimal schedules."""
    case = result["case"]
    x0 = case.x0_grid
    tx0 = result["tx0"]
    y_id = flow_positions(x0, tx0, identity_schedule, T_GRID)
    y_opt = flow_positions(x0, tx0, result["schedule_opt"], T_GRID)

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True, constrained_layout=True)
    for ax, y, title in [
        (axes[0], y_id, "identity schedule"),
        (axes[1], y_opt, "optimal schedule"),
    ]:
        # Same trajectories, different time parameterizations.
        ax.plot(T_GRID, y, color=BLUE if title.startswith("identity") else RED, lw=1.0, alpha=0.75)
        ax.set_title(title)
        ax.set_xlabel("t")
        ax.set_ylabel("X_tau(x0, t)")
    fig.suptitle(f"{case.name}: trajectories")
    plt.show()


def plot_schedule(result):
    """Plot tau(t) for the identity and optimal schedules."""
    tau_opt, tau_dot_opt = result["schedule_opt"](T_GRID)
    fig, ax = plt.subplots(figsize=(5.5, 3.5))
    ax.plot(T_GRID, T_GRID, color=BLUE, lw=1.8, label="tau(t)=t")
    ax.plot(T_GRID, tau_opt, color=RED, lw=2.0, ls="--", label="tau_infty(t)")
    ax.set_title(f"{result['case'].name}: schedule")
    ax.set_xlabel("t")
    ax.set_ylabel("tau(t)")
    ax.legend()
    plt.show()


def plot_lipschitz(result):
    """Plot semilog spatial Lipschitz curves for identity and optimal schedules."""
    fig, ax = plt.subplots(figsize=(6, 3.8))
    ax.semilogy(T_GRID, result["l_id"], color=BLUE, lw=2.0, label="identity")
    ax.semilogy(T_GRID, result["l_opt"], color=RED, lw=2.0, ls="--", label="tau_infty")
    ax.set_title(f"{result['case'].name}: spatial Lipschitz constant")
    ax.set_xlabel("t")
    ax.set_ylabel("L_tau(t)")
    ax.legend()
    plt.show()


def show_case(result):
    """Display the standard table and four diagnostic plots for a case."""
    extrema_table(result)
    plot_tprime(result)
    plot_trajectories(result)
    plot_schedule(result)
    plot_lipschitz(result)


## 1. Gaussian to Gaussian

This is the linear sanity check from Section 4.1. The general 1D rearrangement should collapse to

$$
T(x)=\mu_2 + \frac{\theta_2}{\theta_1}(x-\mu_1),
$$

and the closed-form schedule should be $\tau_\infty(t)=((\theta_2/\theta_1)^t - 1)/(\theta_2/\theta_1 - 1)$.

In [ ]:
# Linear Gaussian sanity check: the quantile map should be exactly 0.01*x.
source_g = scipy_distribution("N(0, 1)", stats.norm(loc=0.0, scale=1.0))
target_tight_g = scipy_distribution("N(0, 0.01^2)", stats.norm(loc=0.0, scale=0.01))

case_gaussian = CaseSpec(
    name="Gaussian -> Gaussian",
    source=source_g,
    target=target_tight_g,
    x_grid=np.linspace(-4.0, 4.0, 4001),
    x0_grid=np.linspace(-3.0, 3.0, 31),
    mode_y=(0.0,),
    gaussian_ratio=0.01,
)

result_gaussian = run_case(case_gaussian)
affine_T = 0.01 * case_gaussian.x_grid
max_affine_error = np.max(np.abs(result_gaussian["tx_plot"] - affine_T))
print(f"max |F_nu^-1(F_mu(x)) - 0.01 x| on plot grid: {max_affine_error:.3e}")
show_case(result_gaussian)


The Gaussian-to-Gaussian case is the calibration point. The derivative plot should be essentially constant at the contraction ratio, while the identity Lipschitz curve grows sharply near $t=1$. The optimal schedule slows down as the contraction becomes stiff, so the semilog Lipschitz plot should look nearly horizontal rather than endpoint-dominated.


## 2. Gaussian to bimodal Gaussian mixture

This matches the paper's bimodal example: $\nu=0.8\,N(-2,0.02^2)+0.2\,N(2,0.01^2)$. The evaluation grid contains a dense Gaussian quantile grid, the two target modes, and extra points around the mixture split at cumulative mass $0.8$. Those split-neighborhood points are what make the expansion side of $T'$ visible on a finite grid.

In [ ]:
# The mixture example needs custom inverse CDFs, supplied by GaussianMixture1D.
mixture = GaussianMixture1D(
    weights=[0.8, 0.2],
    means=[-2.0, 2.0],
    stds=[0.02, 0.01],
    name="0.8 N(-2, 0.02^2) + 0.2 N(2, 0.01^2)",
).as_distribution

# Add dense probability probes around the mass split at u=0.8.
split_offsets = np.r_[np.logspace(-2, -10, 25), np.array([1e-12])]
split_u = np.r_[0.8 - split_offsets, 0.8 + split_offsets]
base_x = quantile_grid(source_g, 1e-5, 1.0 - 1e-5, 5001)
case_mixture = CaseSpec(
    name="Gaussian -> bimodal mixture",
    source=source_g,
    target=mixture,
    x_grid=sorted_unique(np.r_[base_x, source_g.ppf(clip_prob(split_u))]),
    x0_grid=quantile_grid(source_g, 0.01, 0.99, 41),
    mode_y=(-2.0, 2.0),
    extra_u=tuple(split_u),
    log_clip=50.0,
)

result_mixture = run_case(case_mixture)
show_case(result_mixture)


The bimodal target creates both contraction and expansion regions in the monotone map. The transport-derivative plot localizes where the map is hardest, and the Lipschitz plot shows whether $\tau_\infty$ spreads that difficulty across time instead of allowing one endpoint or transition region to dominate.


## 3. Gaussian to Laplace

This case keeps the source Gaussian but uses a heavier-tailed Laplace target. On a compact high-probability grid, the transport is mildly contractive near the center and expansive in the tails.

In [ ]:
# Tail quantiles make the Gaussian-to-Laplace expansion visible on a finite grid.
target_laplace = scipy_distribution("Laplace(0, 1)", stats.laplace(loc=0.0, scale=1.0))
tail_u = np.r_[np.logspace(-6, -3, 16), 1.0 - np.logspace(-6, -3, 16)]
case_laplace = CaseSpec(
    name="Gaussian -> Laplace",
    source=source_g,
    target=target_laplace,
    x_grid=sorted_unique(np.r_[quantile_grid(source_g, 1e-6, 1.0 - 1e-6, 5001), source_g.ppf(tail_u)]),
    x0_grid=quantile_grid(source_g, 0.01, 0.99, 41),
    mode_y=(0.0,),
    extra_u=tuple(tail_u),
)

result_laplace = run_case(case_laplace)
show_case(result_laplace)


For the Laplace target, the tail expansion is milder than in the sharply concentrated examples. The main thing to notice is whether the optimal curve still reduces the largest time-localized Lipschitz values while preserving the same exact endpoint map.


## 4. Uniform to peaked Gaussian

The source is compact, $U(-3,3)$, while the target is sharply concentrated, $N(0,0.05^2)$. The numerical grid avoids the exact uniform endpoints because the Gaussian quantile map diverges at probabilities 0 and 1.

In [ ]:
# Avoid exact uniform endpoints because the Gaussian target quantile is infinite there.
source_uniform = scipy_distribution("U(-3, 3)", stats.uniform(loc=-3.0, scale=6.0))
target_peak = scipy_distribution("N(0, 0.05^2)", stats.norm(loc=0.0, scale=0.05))
endpoint_u = np.r_[np.logspace(-8, -3, 26), 1.0 - np.logspace(-8, -3, 26)]
case_uniform = CaseSpec(
    name="Uniform -> peaked Gaussian",
    source=source_uniform,
    target=target_peak,
    x_grid=sorted_unique(np.r_[quantile_grid(source_uniform, 1e-6, 1.0 - 1e-6, 5001), source_uniform.ppf(endpoint_u)]),
    x0_grid=np.linspace(-2.95, 2.95, 41),
    mode_y=(0.0,),
    extra_u=tuple(endpoint_u),
)

result_uniform = run_case(case_uniform)
show_case(result_uniform)


The uniform-to-peaked-Gaussian map is strongly contractive over most of the source support. This is another endpoint-stiff example: under the identity schedule the denominator in the Lipschitz formula becomes small near the final time, while the optimal schedule should redistribute the contraction over the interval.


## Summary

The table reports the empirical uniform-in-time Lipschitz constant on each case's finite evaluation grid. The headline check is the ratio: values much larger than one mean the optimal schedule substantially reduces the worst-case spatial Lipschitz constant.

In [ ]:
summary = pd.DataFrame(SUMMARY_ROWS)
summary = summary[
    [
        "case",
        "Lambda[id]",
        "Lambda[tau_infty]",
        "ratio",
        "sigma_min",
        "sigma_max",
        "schedule_case",
        "t0",
    ]
]

display(
    summary.style.format(
        {
            "Lambda[id]": "{:.4g}",
            "Lambda[tau_infty]": "{:.4g}",
            "ratio": "{:.4g}",
            "sigma_min": "{:.4g}",
            "sigma_max": "{:.4g}",
            "t0": "{:.4f}",
        }
    )
)


The ratio column is a worst-case diagnostic: it compares only $\max_t L_\tau(t)$. A large value means the optimal schedule reduced the empirical peak stiffness, but it does not by itself say whether the whole curve became flatter. The next table adds distributional summaries of $L_\tau(t)$ over time.


## Lipschitz diagnostics

The semilog plots show the mechanism visually; here we make it quantitative. For each schedule we summarize the empirical time curve $L(t)$ by its maximum, mean, median, standard deviation, coefficient of variation, and endpoint blow-up ratio.

The endpoint blow-up ratio compares the largest Lipschitz value near the ends to a typical interior value:

$$
\frac{\max_{t\in[0,0.05]\cup[0.95,1]} L(t)}
     {\operatorname{median}_{t\in[0.1,0.9]} L(t)}.
$$

Thus the table measures both worst-case stiffness and flattening. Reductions larger than one favor $\tau_\infty$ over the identity schedule.


In [ ]:
def _safe_ratio(numerator: float, denominator: float, eps: float = 1e-12) -> float:
    """Divide two scalar diagnostics while guarding against zero denominators."""
    numerator = float(numerator)
    denominator = float(denominator)
    if not np.isfinite(numerator) or not np.isfinite(denominator):
        return np.nan
    return numerator / max(abs(denominator), eps)


def _lipschitz_summary(t_grid: np.ndarray, values: np.ndarray, eps: float = 1e-12) -> dict:
    """Summarize one empirical Lipschitz curve by peak, spread, and endpoints."""
    t_grid = np.asarray(t_grid, dtype=float)
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    finite_values = values[finite]
    if finite_values.size == 0:
        return {
            "Lambda": np.nan,
            "mean_L": np.nan,
            "median_L": np.nan,
            "std_L": np.nan,
            "flatness_cv": np.nan,
            "endpoint_max": np.nan,
            "interior_median": np.nan,
            "endpoint_blowup": np.nan,
        }

    # Endpoint and interior windows quantify where stiffness concentrates in time.
    endpoint_mask = finite & ((t_grid <= 0.05) | (t_grid >= 0.95))
    interior_mask = finite & (t_grid >= 0.1) & (t_grid <= 0.9)
    endpoint_values = values[endpoint_mask]
    interior_values = values[interior_mask]

    mean_l = float(np.mean(finite_values))
    std_l = float(np.std(finite_values))
    endpoint_max = float(np.max(endpoint_values)) if endpoint_values.size else np.nan
    interior_median = float(np.median(interior_values)) if interior_values.size else np.nan

    return {
        "Lambda": float(np.max(finite_values)),
        "mean_L": mean_l,
        "median_L": float(np.median(finite_values)),
        "std_L": std_l,
        "flatness_cv": _safe_ratio(std_l, mean_l, eps),
        "endpoint_max": endpoint_max,
        "interior_median": interior_median,
        "endpoint_blowup": _safe_ratio(endpoint_max, interior_median, eps),
    }


def lipschitz_diagnostics(
    t_grid: np.ndarray,
    l_id: np.ndarray,
    l_opt: np.ndarray,
) -> dict:
    """Compare identity and optimal Lipschitz curves in one diagnostic row."""
    id_stats = _lipschitz_summary(t_grid, l_id)
    opt_stats = _lipschitz_summary(t_grid, l_opt)

    row = {}
    for key, value in id_stats.items():
        row[f"{key}_id"] = value
    for key, value in opt_stats.items():
        row[f"{key}_opt"] = value

    # Ratios above one mean the optimal schedule improved the corresponding metric.
    row["Lambda_reduction"] = _safe_ratio(row["Lambda_id"], row["Lambda_opt"])
    row["mean_reduction"] = _safe_ratio(row["mean_L_id"], row["mean_L_opt"])
    row["flatness_cv_reduction"] = _safe_ratio(row["flatness_cv_id"], row["flatness_cv_opt"])
    row["endpoint_blowup_reduction"] = _safe_ratio(row["endpoint_blowup_id"], row["endpoint_blowup_opt"])
    return row


In [ ]:
lipschitz_rows = []
for result in CASE_RESULTS:
    diagnostics = lipschitz_diagnostics(T_GRID, result["l_id"], result["l_opt"])
    diagnostics["case"] = result["case"].name
    lipschitz_rows.append(diagnostics)

lipschitz_df = pd.DataFrame(lipschitz_rows)

# Keep the displayed table focused on the comparison metrics discussed above.
lipschitz_table = lipschitz_df[
    [
        "case",
        "Lambda_id",
        "Lambda_opt",
        "Lambda_reduction",
        "flatness_cv_id",
        "flatness_cv_opt",
        "flatness_cv_reduction",
        "endpoint_blowup_id",
        "endpoint_blowup_opt",
        "endpoint_blowup_reduction",
    ]
]

display(
    lipschitz_table.style.format(
        {
            "Lambda_id": "{:.4g}",
            "Lambda_opt": "{:.4g}",
            "Lambda_reduction": "{:.4g}",
            "flatness_cv_id": "{:.4g}",
            "flatness_cv_opt": "{:.4g}",
            "flatness_cv_reduction": "{:.4g}",
            "endpoint_blowup_id": "{:.4g}",
            "endpoint_blowup_opt": "{:.4g}",
            "endpoint_blowup_reduction": "{:.4g}",
        }
    )
)


The peak reduction columns say whether the hardest instant became easier. The coefficient-of-variation and endpoint-blow-up columns say whether the stiffness was redistributed over time. When both improve, the optimal schedule is not merely lowering one spike; it is making the ODE more uniformly conditioned across the interval.


## Wasserstein integration diagnostics

Since both schedules reach $T$ exactly, Wasserstein distance only distinguishes them after we deliberately introduce numerical time discretization. At a fixed number of solver steps, a smaller final Wasserstein error indicates that the scheduled ODE was easier to integrate.

To remove Monte Carlo noise, the source samples are deterministic quantiles rather than random draws. If $u_i$ are evenly spaced in $(0,1)$, then

$$
x_i = F_\mu^{-1}(u_i), \qquad x_i^{\star}=T(x_i)=F_\nu^{-1}(u_i).
$$

The finite-step solver evolves the particles and we compare its final empirical distribution to the exact target-matched quantile samples $x_i^{\star}$. The implementation uses interior quantiles so the particles stay inside the compact numerical reference grids used for interpolation.


The ODE velocity is naturally known in reference coordinates:

$$
y=X_\tau(x,t)=(1-\tau(t))x+\tau(t)T(x),
\qquad
v_\tau(y,t)=\dot\tau(t)(T(x)-x).
$$

A numerical solver, however, stores the current positions $y$. At each stage we invert the one-dimensional map $x\mapsto X_\tau(x,t)$ by interpolation on the same monotone reference grid used above, then interpolate the displacement $T(x)-x$. This keeps the diagnostic analytical and 1D-only while still testing the actual Euler and RK4 time stepping problem.


In [ ]:
def deterministic_source_samples(
    source: Distribution1D,
    n: int,
    eps: float = 1e-6,
) -> np.ndarray:
    """Return deterministic interior source quantiles for ODE diagnostics."""
    u = np.linspace(eps, 1.0 - eps, n)
    return np.asarray(source.ppf(clip_prob(u)), dtype=float)


def velocity_at_y(
    y: np.ndarray,
    t: float,
    x_ref: np.ndarray,
    tx_ref: np.ndarray,
    schedule,
    time_eps: float = 1e-10,
) -> np.ndarray:
    """Evaluate v_tau(y,t) by inverting X_tau(x,t) on a 1D reference grid.

    The analytical velocity is known as tau_dot(t) * (T(x)-x), but the solver
    stores current positions y. In one dimension, X_tau(.,t) is monotone, so
    interpolation gives a stable approximate inverse x = X_tau^{-1}(y,t).
    """
    y = np.asarray(y, dtype=float)
    x_ref = np.asarray(x_ref, dtype=float)
    tx_ref = np.asarray(tx_ref, dtype=float)

    # Avoid evaluating schedules exactly at singular or numerically stiff endpoints.
    t_eval = float(np.clip(t, time_eps, 1.0 - time_eps))
    tau, tau_dot = schedule(np.asarray([t_eval], dtype=float))
    tau_t = float(tau[0])
    tau_dot_t = float(tau_dot[0])

    # Build the forward map y_ref = X_tau(x_ref,t), then sort for np.interp.
    y_ref = (1.0 - tau_t) * x_ref + tau_t * tx_ref
    order = np.argsort(y_ref)
    y_sorted = y_ref[order]
    x_sorted = x_ref[order]
    finite = np.isfinite(y_sorted) & np.isfinite(x_sorted)
    y_sorted = y_sorted[finite]
    x_sorted = x_sorted[finite]

    # Remove duplicate y values, which can appear after rounding or extreme contraction.
    y_unique, unique_idx = np.unique(y_sorted, return_index=True)
    x_unique = x_sorted[unique_idx]
    if y_unique.size < 2:
        raise ValueError("Reference map is not invertible on the numerical grid.")

    # Clip-style extrapolation at the boundaries keeps the solver finite.
    x_inv = np.interp(y, y_unique, x_unique, left=x_unique[0], right=x_unique[-1])
    displacement_ref = tx_ref - x_ref
    displacement = np.interp(
        x_inv,
        x_ref,
        displacement_ref,
        left=displacement_ref[0],
        right=displacement_ref[-1],
    )
    return tau_dot_t * displacement


def make_velocity_interpolator(
    case: CaseSpec,
    schedule,
    x_ref: np.ndarray,
    tx_ref: np.ndarray | None = None,
):
    """Create a reusable velocity callable for a case, schedule, and grid."""
    if tx_ref is None:
        tx_ref, _, _ = transport_values(case, x_ref)

    def velocity(y: np.ndarray, t: float) -> np.ndarray:
        """Evaluate the cached-grid velocity at solver state y and time t."""
        return velocity_at_y(y, t, x_ref, tx_ref, schedule)

    return velocity


def integrate_ode(
    y0: np.ndarray,
    schedule,
    case: CaseSpec,
    n_steps: int,
    method: str = "rk4",
    x_ref: np.ndarray | None = None,
    tx_ref: np.ndarray | None = None,
) -> np.ndarray:
    """Integrate the scheduled ODE with fixed-step Euler or classical RK4."""
    if n_steps <= 0:
        raise ValueError("n_steps must be positive.")
    if method not in {"euler", "rk4"}:
        raise ValueError("method must be 'euler' or 'rk4'.")

    if x_ref is None:
        x_ref = make_evaluation_grid(case)
    if tx_ref is None:
        tx_ref, _, _ = transport_values(case, x_ref)

    velocity = make_velocity_interpolator(case, schedule, x_ref, tx_ref)
    y = np.asarray(y0, dtype=float).copy()
    dt = 1.0 / float(n_steps)

    for k in range(n_steps):
        t = k * dt
        if method == "euler":
            # First-order explicit step from the left endpoint of the time interval.
            y = y + dt * velocity(y, t)
        else:
            # Classical RK4: four velocity evaluations per fixed time step.
            k1 = velocity(y, t)
            k2 = velocity(y + 0.5 * dt * k1, t + 0.5 * dt)
            k3 = velocity(y + 0.5 * dt * k2, t + 0.5 * dt)
            k4 = velocity(y + dt * k3, t + dt)
            y = y + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

    return y


For equal-weight empirical distributions in one dimension, Wasserstein distances are quantile distances. Sorting both samples gives

$$
W_1(a,b)=\frac{1}{n}\sum_{i=1}^n |a_{(i)}-b_{(i)}|,
\qquad
W_2(a,b)=\left(\frac{1}{n}\sum_{i=1}^n |a_{(i)}-b_{(i)}|^2\right)^{1/2}.
$$

`scipy.stats.wasserstein_distance` implements $W_1$; here both $W_1$ and $W_2$ are implemented directly from the one-dimensional quantile formula so that the two diagnostics use the same sorted empirical samples.


In [ ]:
def wasserstein_w1_sorted(a: np.ndarray, b: np.ndarray) -> float:
    """Compute equal-weight empirical W1 in 1D by sorting both samples."""
    a_sorted = np.sort(np.asarray(a, dtype=float))
    b_sorted = np.sort(np.asarray(b, dtype=float))
    if a_sorted.shape != b_sorted.shape:
        raise ValueError("Empirical samples must have the same length.")
    return float(np.mean(np.abs(a_sorted - b_sorted)))


def wasserstein_w2_sorted(a: np.ndarray, b: np.ndarray) -> float:
    """Compute equal-weight empirical W2 in 1D by sorting both samples."""
    a_sorted = np.sort(np.asarray(a, dtype=float))
    b_sorted = np.sort(np.asarray(b, dtype=float))
    if a_sorted.shape != b_sorted.shape:
        raise ValueError("Empirical samples must have the same length.")
    return float(np.sqrt(np.mean((a_sorted - b_sorted) ** 2)))


In [ ]:
n_samples = 2000
sample_eps = 1e-4
step_counts = [8, 16, 32, 64, 128]
methods = ["euler", "rk4"]

wasserstein_rows = []
for result in CASE_RESULTS:
    case = result["case"]

    # Deterministic quantiles remove Monte Carlo noise from the solver comparison.
    x0 = deterministic_source_samples(case.source, n_samples, eps=sample_eps)
    x_target_exact, _, _ = transport_values(case, x0)
    target_sorted = np.sort(x_target_exact)

    # Reuse one enriched reference grid per case for all schedules and step counts.
    x_ref = make_evaluation_grid(case)
    tx_ref, _, _ = transport_values(case, x_ref)
    schedules = {
        "id": identity_schedule,
        "opt": result["schedule_opt"],
    }

    for method in methods:
        for n_steps in step_counts:
            for schedule_name, schedule in schedules.items():
                y_final = integrate_ode(
                    x0,
                    schedule,
                    case,
                    n_steps=n_steps,
                    method=method,
                    x_ref=x_ref,
                    tx_ref=tx_ref,
                )
                y_sorted = np.sort(y_final)
                abs_errors = np.abs(y_sorted - target_sorted)
                wasserstein_rows.append(
                    {
                        "case": case.name,
                        "method": method,
                        "n_steps": n_steps,
                        "schedule": schedule_name,
                        "W1": wasserstein_w1_sorted(y_final, x_target_exact),
                        "W2": wasserstein_w2_sorted(y_final, x_target_exact),
                        "max_abs_error": float(np.max(abs_errors)),
                        "mean_abs_error": float(np.mean(abs_errors)),
                    }
                )

wasserstein_df = pd.DataFrame(wasserstein_rows)

# Pivot to compare schedules side-by-side at each method and step budget.
wasserstein_comparison = wasserstein_df.pivot_table(
    index=["case", "method", "n_steps"],
    columns="schedule",
    values=["W1", "W2"],
).reset_index()
wasserstein_comparison.columns = [
    "_".join([str(part) for part in col if str(part)])
    if isinstance(col, tuple)
    else str(col)
    for col in wasserstein_comparison.columns
]
wasserstein_comparison["W1_ratio_id_over_opt"] = wasserstein_comparison["W1_id"] / np.maximum(
    wasserstein_comparison["W1_opt"], 1e-15
)
wasserstein_comparison["W2_ratio_id_over_opt"] = wasserstein_comparison["W2_id"] / np.maximum(
    wasserstein_comparison["W2_opt"], 1e-15
)
wasserstein_comparison = wasserstein_comparison[
    [
        "case",
        "method",
        "n_steps",
        "W1_id",
        "W1_opt",
        "W1_ratio_id_over_opt",
        "W2_id",
        "W2_opt",
        "W2_ratio_id_over_opt",
    ]
]

display(
    wasserstein_comparison.style.format(
        {
            "W1_id": "{:.4e}",
            "W1_opt": "{:.4e}",
            "W1_ratio_id_over_opt": "{:.3g}",
            "W2_id": "{:.4e}",
            "W2_opt": "{:.4e}",
            "W2_ratio_id_over_opt": "{:.3g}",
        }
    )
)


This table should be read as a finite-step solver diagnostic, not as a property of the exact transport. Higher ratios mean the identity schedule produced larger final distribution error than $\tau_\infty$ at the same solver, step count, and deterministic sample set. Lower W1/W2 at fixed step budget indicates an easier ODE to integrate.

There is one important caveat in this exact reference setting: for $\tau(t)=t$, each characteristic is affine in time, so explicit Euler and RK stages can be nearly exact when the inverse map is evaluated accurately. When the identity error sits at the interpolation floor, the Wasserstein diagnostic correctly reports that there is little finite-step error left for the optimal schedule to reduce.


In [ ]:
def plot_w2_by_steps(wasserstein_df: pd.DataFrame, method: str = "rk4"):
    """Plot final W2 error versus step count for each case and schedule."""
    for case_name in wasserstein_df["case"].drop_duplicates():
        fig, ax = plt.subplots(figsize=(6, 3.8))
        for schedule_name, color, label in [
            ("id", BLUE, "identity"),
            ("opt", RED, "tau_infty"),
        ]:
            subset = wasserstein_df[
                (wasserstein_df["case"] == case_name)
                & (wasserstein_df["method"] == method)
                & (wasserstein_df["schedule"] == schedule_name)
            ].sort_values("n_steps")
            ax.loglog(
                subset["n_steps"],
                subset["W2"],
                marker="o",
                color=color,
                lw=2.0,
                label=label,
            )
        ax.set_title(f"{case_name}: {method.upper()} final W2 error")
        ax.set_xlabel("number of ODE steps")
        ax.set_ylabel("final W2 error")
        ax.legend()
        plt.show()


plot_w2_by_steps(wasserstein_df, method="rk4")


The RK4 curves isolate the integration-quality question visually. Downward slopes indicate convergence as the step budget increases. A lower red curve means the same RK4 method and number of steps were more accurate under $\tau_\infty$; a lower blue curve means the identity schedule was already easier for that exact solver. Either outcome is informative because the exact endpoint map is the same in both cases.


## Numerical checks

The assertions below are intentionally lightweight. They verify the main analytical and numerical invariants without requiring every schedule to win for every case, solver, and step count. The exact endpoint is schedule-invariant, but the finite-step errors should be finite, non-negative, and generally improve as RK4 receives more steps.

For the Gaussian contraction, the checks also record the affine-characteristic fact: with the exact inverse velocity, the identity schedule is essentially at interpolation precision, while the nonlinear optimal schedule should converge rapidly as the step count increases.


In [ ]:
for result in CASE_RESULTS:
    case = result["case"]
    diagnostics = lipschitz_diagnostics(T_GRID, result["l_id"], result["l_opt"])
    assert diagnostics["Lambda_opt"] <= 1.05 * diagnostics["Lambda_id"], case.name

stiff_cases = {
    "Gaussian -> Gaussian",
    "Gaussian -> bimodal mixture",
    "Uniform -> peaked Gaussian",
}
for result in CASE_RESULTS:
    case = result["case"]
    if case.name in stiff_cases:
        diagnostics = lipschitz_diagnostics(T_GRID, result["l_id"], result["l_opt"])
        assert diagnostics["flatness_cv_opt"] < diagnostics["flatness_cv_id"], case.name

for result in CASE_RESULTS:
    case = result["case"]
    if case.gaussian_ratio is not None:
        expected_tx = case.gaussian_ratio * case.x_grid
        max_error = float(np.max(np.abs(result["tx_plot"] - expected_tx)))
        diagnostics = lipschitz_diagnostics(T_GRID, result["l_id"], result["l_opt"])
        assert max_error < 1e-8
        assert diagnostics["flatness_cv_opt"] < 1e-3

error_columns = ["W1", "W2", "max_abs_error", "mean_abs_error"]
assert np.all(np.isfinite(wasserstein_df[error_columns].to_numpy()))
assert np.all(wasserstein_df[error_columns].to_numpy() >= 0.0)

for (case_name, schedule_name), subset in wasserstein_df[wasserstein_df["method"] == "rk4"].groupby(["case", "schedule"]):
    ordered = subset.sort_values("n_steps")
    w2_values = ordered["W2"].to_numpy()
    assert w2_values[-1] <= 1.05 * w2_values[0] + 1e-12, (case_name, schedule_name)

gaussian_rk4 = wasserstein_comparison[
    (wasserstein_comparison["case"] == "Gaussian -> Gaussian")
    & (wasserstein_comparison["method"] == "rk4")
].sort_values("n_steps")
gaussian_w2_id = gaussian_rk4["W2_id"].to_numpy()
gaussian_w2_opt = gaussian_rk4["W2_opt"].to_numpy()
assert gaussian_w2_id[0] < 1e-6
assert gaussian_w2_opt[-1] < 0.05 * gaussian_w2_opt[0]

print("All numerical checks passed.")


## Conclusion

Across these one-dimensional reference cases, $\tau_\infty$ flattens the empirical spatial Lipschitz profile: the peak, coefficient-of-variation, and endpoint-blow-up diagnostics quantify the same mechanism shown by the semilog plots.

The Wasserstein integration test is complementary rather than redundant. It does not compare exact endpoints, which are schedule-invariant. It compares finite-step ODE solutions against exact deterministic target quantiles. In this exact 1D reference implementation, the identity schedule can be nearly exact because its characteristics are affine in time, so $\tau_\infty$ does not have to win every finite-step Wasserstein comparison. Where the optimal schedule has larger error at coarse step counts, the convergence curves show whether that error shrinks as the solver budget increases.

Together, the diagnostics separate mechanism from consequence: Lipschitz flattening measures reduced spatial stiffness, while Wasserstein error measures what a particular finite-step solver actually does with the scheduled velocity field.
